In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # Bronze to Silver
# MAGIC Notebook responsável por ler as tabelas da camada Bronze, aplicar limpeza, tipagem, deduplicação e regras de negócio para a camada Silver.

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Criar a base de dados da camada Silver se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS silver;")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 1) silver.tb_cotacao_dolar

# COMMAND ----------
df_cotacao = spark.read.table("bronze.tb_cotacao_dolar")

df_cotacao = df_cotacao.withColumn("data_ref", F.to_date(F.substring("dataHoraCotacao", 1, 10))) \
                       .withColumn("cotacaoCompra", F.expr("try_cast(cotacaoCompra as double)"))

limites = df_cotacao.agg(F.min("data_ref").alias("min_dt"), F.max("data_ref").alias("max_dt")).collect()[0]
min_dt = limites["min_dt"]
max_dt = limites["max_dt"]

if min_dt and max_dt:
    df_datas = spark.sql(f"SELECT explode(sequence(to_date('{min_dt}'), to_date('{max_dt}'), interval 1 day)) as data_cotacao")
    df_cotacao_joined = df_datas.join(df_cotacao, df_datas.data_cotacao == df_cotacao.data_ref, "left")
    window_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
    df_silver_cotacao = df_cotacao_joined.withColumn("cotacao_venda", F.last("cotacaoCompra", ignorenulls=True).over(window_ffill)) \
                                         .select("data_cotacao", "cotacao_venda")
else:
    df_silver_cotacao = spark.createDataFrame([], schema="data_cotacao date, cotacao_venda double")

df_silver_cotacao.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")
print("Tabela silver.tb_cotacao_dolar gravada com sucesso.")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 2) silver.tb_info_filmes

# COMMAND ----------
df_info = spark.read.table("bronze.tb_movies_info")

df_info = df_info.withColumnRenamed("id", "id_filme") \
                 .withColumnRenamed("title", "titulo") \
                 .withColumnRenamed("original_title", "titulo_original") \
                 .withColumnRenamed("release_date", "data_lancamento") \
                 .withColumnRenamed("runtime", "duracao_minutos") \
                 .withColumnRenamed("original_language", "idioma_original") \
                 .withColumnRenamed("status", "status_filme") \
                 .withColumnRenamed("overview", "sinopse") \
                 .withColumnRenamed("tagline", "frase_divulgacao")

window_dedup = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_info = df_info.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

# Tratamento de Data Multi-Formato com try_to_date
formatos_data = ["yyyy-MM-dd", "dd/MM/yyyy", "MM-dd-yyyy", "yyyy/MM/dd"]
expr_datas = [F.expr(f"try_to_date(data_lancamento, '{fmt}')") for fmt in formatos_data]
df_info = df_info.withColumn("data_lancamento", F.coalesce(*expr_datas))

df_info = df_info.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

df_info = df_info.withColumn("status_norm", F.lower(F.trim(F.regexp_replace("status_filme", "[-_]", " "))))

mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado"
}
expr_mapa = F.create_map([F.lit(x) for kv in mapa_status.items() for x in kv])
df_info = df_info.withColumn("status_filme", F.coalesce(expr_mapa[F.col("status_norm")], F.lit("Não Informado")))

# CORREÇÃO: Utilizando try_cast para tolerar strings do Column Shift
df_info = df_info.withColumn("duracao_minutos", F.expr("try_cast(duracao_minutos as int)"))

df_silver_info = df_info.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
)
df_silver_info.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")
print("Tabela silver.tb_info_filmes gravada com sucesso.")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 3) silver.tb_financeiro_filmes

# COMMAND ----------
df_fin = spark.read.table("bronze.tb_movies_financials")
df_fin = df_fin.withColumnRenamed("id", "id_filme") \
               .withColumnRenamed("budget", "orcamento_usd") \
               .withColumnRenamed("revenue", "receita_usd")

colunas_fin = ["orcamento_usd", "receita_usd"]
for c in colunas_fin:
    df_fin = df_fin.withColumn(c, F.when(F.col(c).isin("Unknown", "Não Informado", "", "null"), F.lit(None)).otherwise(F.col(c)))
    df_fin = df_fin.withColumn(c, F.regexp_replace(F.col(c), r"[^\d\.]", ""))
    # CORREÇÃO: try_cast para evitar erro na conversão
    df_fin = df_fin.withColumn(c, F.expr(f"try_cast({c} as decimal(18,2))"))
    df_fin = df_fin.withColumn(c, F.when(F.col(c) <= 0, F.lit(None)).otherwise(F.col(c)))

df_info_datas = spark.read.table("silver.tb_info_filmes").select("id_filme", "data_lancamento")
df_fin = df_fin.join(df_info_datas, "id_filme", "left")

df_silver_cotacao = spark.read.table("silver.tb_cotacao_dolar")
taxa_fallback_row = df_silver_cotacao.orderBy(F.col("data_cotacao").desc()).limit(1).collect()
taxa_fallback = taxa_fallback_row[0]["cotacao_venda"] if taxa_fallback_row else None

df_fin = df_fin.join(df_silver_cotacao, df_fin.data_lancamento == df_silver_cotacao.data_cotacao, "left")
df_fin = df_fin.withColumn("taxa_aplicada", F.coalesce(F.col("cotacao_venda"), F.lit(taxa_fallback)))

df_fin = df_fin.withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("taxa_aplicada")).cast("decimal(18,2)"))
df_fin = df_fin.withColumn("receita_brl", (F.col("receita_usd") * F.col("taxa_aplicada")).cast("decimal(18,2)"))
df_fin = df_fin.withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
df_fin = df_fin.withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))

df_fin = df_fin.withColumn(
    "margem_lucro_percentual",
    F.when(F.col("receita_usd").isNotNull() & (F.col("receita_usd") > 0), 
           ((F.col("lucro_usd") / F.col("receita_usd")) * 100).cast("decimal(18,2)"))
     .otherwise(F.lit(None))
)

df_silver_fin = df_fin.select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_percentual")
df_silver_fin.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")
print("Tabela silver.tb_financeiro_filmes gravada com sucesso.")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 4) silver.tb_metricas_engajamento

# COMMAND ----------
df_met = spark.read.table("bronze.tb_movies_metrics")
df_met = df_met.withColumnRenamed("id", "id_filme") \
               .withColumnRenamed("popularity", "popularidade") \
               .withColumnRenamed("vote_average", "nota_media_tmdb") \
               .withColumnRenamed("vote_count", "qtd_votos_tmdb") \
               .withColumnRenamed("averageRating", "nota_media_imdb") \
               .withColumnRenamed("numVotes", "qtd_votos_imdb")

df_met = df_met.withColumn("popularidade", F.regexp_replace("popularidade", ",", "."))
# CORREÇÃO: try_cast para lidar com o column shift
df_met = df_met.withColumn("popularidade", F.expr("try_cast(popularidade as double)"))
df_met = df_met.withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")).otherwise(F.lit(None)))

for c in ["nota_media_tmdb", "nota_media_imdb"]:
    # CORREÇÃO: try_cast
    df_met = df_met.withColumn(c, F.expr(f"try_cast({c} as double)"))
    df_met = df_met.withColumn(c, F.when((F.col(c) >= 0) & (F.col(c) <= 10), F.col(c)).otherwise(F.lit(None)))

for c in ["qtd_votos_tmdb", "qtd_votos_imdb"]:
    # CORREÇÃO: try_cast
    df_met = df_met.withColumn(c, F.expr(f"try_cast({c} as int)"))
    df_met = df_met.withColumn(c, F.when(F.col(c) >= 0, F.col(c)).otherwise(F.lit(None)))

df_silver_met = df_met.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
df_silver_met.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")
print("Tabela silver.tb_metricas_engajamento gravada com sucesso.")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 5) silver.tb_avaliacoes_usuarios

# COMMAND ----------
df_rev = spark.read.table("bronze.tb_movies_reviews")
df_rev = df_rev.withColumnRenamed("id", "id_filme") \
               .withColumnRenamed("nome", "nome_usuario") \
               .withColumnRenamed("nota", "nota_usuario") \
               .withColumnRenamed("comentario", "comentario_usuario")

df_rev = df_rev.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])

# CORREÇÃO: try_cast
df_rev = df_rev.withColumn("nota_usuario", F.expr("try_cast(nota_usuario as double)"))
df_rev = df_rev.withColumn("nota_usuario", F.when((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10), F.col("nota_usuario")).otherwise(F.lit(None)))

df_rev = df_rev.withColumn("comentario_usuario", F.trim(F.col("comentario_usuario")))
df_rev = df_rev.withColumn("comentario_usuario", F.when((F.col("comentario_usuario") == "") | (F.col("comentario_usuario").isNull()), F.lit("Sem comentário")).otherwise(F.col("comentario_usuario")))

df_rev.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")
print("Tabela silver.tb_avaliacoes_usuarios gravada com sucesso.")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 6) silver.tb_generos e silver.tb_pessoas_empresas

# COMMAND ----------
df_cred = spark.read.table("bronze.tb_credits_and_tags").withColumnRenamed("id", "id_filme")

df_gen = df_cred.select("id_filme", "genres")
df_gen = df_gen.withColumn("genres", F.regexp_replace("genres", ";", ","))
df_gen = df_gen.withColumn("nome_genero", F.explode(F.split("genres", ",")))
df_gen = df_gen.withColumn("nome_genero", F.trim("nome_genero"))

# CORREÇÃO: try_cast para tratar sujeira numérica sem travar o ANSI mode
df_gen = df_gen.filter((F.length("nome_genero") > 0) & (F.expr("try_cast(nome_genero as int)").isNull()))
df_gen = df_gen.select("id_filme", "nome_genero").dropDuplicates()

df_gen.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")
print("Tabela silver.tb_generos gravada com sucesso.")

def processar_entidade(df, col_origem, tipo_entidade):
    df_temp = df.select("id_filme", col_origem)
    df_temp = df_temp.withColumn(col_origem, F.regexp_replace(col_origem, ";", ","))
    df_temp = df_temp.withColumn("nome_entidade", F.explode(F.split(col_origem, ",")))
    df_temp = df_temp.withColumn("nome_entidade", F.trim(F.initcap("nome_entidade")))
    
    # CORREÇÃO: try_cast para tratar sujeira numérica
    df_temp = df_temp.filter((F.length("nome_entidade") > 0) & (F.expr("try_cast(nome_entidade as int)").isNull()))
    
    df_temp = df_temp.withColumn("tipo_entidade", F.lit(tipo_entidade))
    return df_temp.select("id_filme", "nome_entidade", "tipo_entidade")

df_atores = processar_entidade(df_cred, "cast", "Ator")
df_diretores = processar_entidade(df_cred, "directors", "Diretor")
df_roteiristas = processar_entidade(df_cred, "writers", "Roteirista")
df_produtoras = processar_entidade(df_cred, "production_companies", "Produtora")

df_pessoas_empresas = df_atores.union(df_diretores).union(df_roteiristas).union(df_produtoras)
df_pessoas_empresas = df_pessoas_empresas.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

df_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")
print("Tabela silver.tb_pessoas_empresas gravada com sucesso.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.tb_cotacao_dolar gravada com sucesso.
Tabela silver.tb_info_filmes gravada com sucesso.
Tabela silver.tb_financeiro_filmes gravada com sucesso.
Tabela silver.tb_metricas_engajamento gravada com sucesso.
Tabela silver.tb_avaliacoes_usuarios gravada com sucesso.
Tabela silver.tb_generos gravada com sucesso.
Tabela silver.tb_pessoas_empresas gravada com sucesso.
